# The full benchmark — 10 models + HAR-RV × 10 datasets × h = 1, 5, 22

Trains every deep model on all five forex and all five crypto realized-variance
series at horizons 1, 5 and 22, fits HAR-RV on the same rows, and writes

* the metric tables (MSE, MAE, QLIKE, on both the `ln(RV)` and the variance scale), and
* the **per-observation loss series** that Diebold–Mariano and the MCS need.

Hyper-parameters come from the Optuna winners in `tuning/ProjectC_tuning`.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (or better).

Run the cells in order. Steps 2–4 are setup, step 6 is a one-minute smoke test,
step 7 is the sweep and is the long one. Everything is written to Google Drive
and the sweep **resumes**: if Colab disconnects, re-run the setup cells and
step 7 again — cells already on Drive are skipped.

## 1. Check the GPU

In [ ]:
!nvidia-smi || echo 'No GPU — switch Runtime > Change runtime type > T4 GPU (300 trainings on CPU is not realistic)'

## 2. Mount Drive

The forecasts, tables and loss matrices go here so they survive a disconnect.
Checkpoints do **not** — they are rewritten every improving epoch, and on a
Drive mount that would dominate the runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/ProjectC_benchmark'   # <- change if you like
CKPT_DIR    = '/content/_ckpt'                              # local disk, deleted per cell

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print('results ->', RESULTS_DIR)

## 3. Clone the repository

`Mr0022/ProjectC` is public, so this needs no credentials. Re-running the cell
in a later session updates an existing clone instead of failing.

Switch `BRANCH` to `'main'` once this work is merged.

In [ ]:
import os, subprocess

BRANCH   = 'claude/orchestrate-model-eval-forex-crypto-rir1dh'
REPO_URL = 'https://github.com/Mr0022/ProjectC.git'
REPO_DIR = '/content/ProjectC'

if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)   # the code resolves models/ and data/ relatively
print(subprocess.run(['git', 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout)

## 4. Install the dependencies Colab is missing

Colab already ships torch, numpy, pandas, scikit-learn, statsmodels and
matplotlib. These are the extras this repo needs — `fast_pytorch_kmeans` for
AdaWaveNet, `reformer-pytorch`/`local-attention` because
`layers/SelfAttention_Family.py` imports them at module load.

In [ ]:
!pip install -q einops PyWavelets fast_pytorch_kmeans reformer-pytorch local-attention psutil

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 5. Validate every configuration (~1 minute, CPU)

Builds all 10 models at all 3 horizons and pushes one batch through each, then
prints how many forecasts each dataset holds per horizon. A configuration the
architecture rejects should surface here, not at hour six of the sweep.

In [ ]:
!python orchestrate/run_benchmark.py --validate

## 6. Smoke test (~2 minutes)

Two cheap models on one dataset at one horizon, 5 epochs, written to local disk
so it cannot be mistaken for a real cell later. If this prints a metrics table,
the environment is wired up correctly.

In [ ]:
!python -u orchestrate/run_benchmark.py \
    --datasets EURUSD --horizons 1 --models DLinear FITS HAR-RV --quick \
    --results_dir /content/_smoke --checkpoint_dir $CKPT_DIR 2>&1 | tail -25

## 7. The sweep — 300 deep-model cells + 10 HAR-RV fits

One subprocess per cell, so a crash costs one cell. Cells already on Drive are
skipped, which is what makes this cell safe to re-run after a disconnect.

Ordered dataset → horizon → model, so an interrupted run leaves **complete**
(dataset, horizon) blocks behind — the unit the MCS is defined over.

Edit `EXTRA` to run a subset, e.g. `['--assets', 'crypto']` or
`['--models', 'FITS', 'DLinear']`. `['--itr', '3']` runs 3 repeats per cell instead of the default 10.

In [ ]:
import subprocess, time

EXTRA = []          # e.g. ['--assets', 'crypto'] or ['--horizons', '1', '5']

started = time.time()
subprocess.run(['python', '-u', 'orchestrate/run_benchmark.py',
                '--results_dir', RESULTS_DIR,
                '--checkpoint_dir', CKPT_DIR] + EXTRA)
print(f'\nsweep finished in {(time.time() - started) / 3600:.2f} h')

## 8. The tables

`aggregate_results.py` re-scores whatever is on Drive — no retraining — so this
cell is also how you look at a sweep that is still running.

In [ ]:
!python orchestrate/aggregate_results.py --results_dir $RESULTS_DIR

In [ ]:
import os
import pandas as pd

summary = pd.read_csv(os.path.join(RESULTS_DIR, 'tables', 'metrics_mean.csv'))
display(summary.head(20))

# models x datasets, one metric, one horizon -- the shape a paper table has
for h in (1, 5, 22):
    path = os.path.join(RESULTS_DIR, 'tables', f'pivot_QLIKE_h{h:02d}.csv')
    if os.path.exists(path):
        print(f'\nQLIKE, h = {h}')
        display(pd.read_csv(path, index_col=0).round(6))

## 9. The DM / MCS inputs

`losses/<dataset>_h<hh>__<loss>__seed<S>.csv` is date-indexed, one column per
model, one row per forecast, for each of `se_ln`, `ae_ln`, `qlike`, `se_rv`,
`ae_rv`. There is one file per seed plus a `__seedmean` one for the forecast each model's repeats average to — run the tests on that, so there is one statistic per model rather than ten that cannot be pooled.

At h > 1 the target windows of consecutive rows overlap by h−1 days, so the
loss differential is autocorrelated by construction: the DM long-run variance
needs a HAC estimator with at least h−1 lags, and the MCS block bootstrap needs
a block length that respects the same overlap.

In [ ]:
import os
import pandas as pd

L = pd.read_csv(os.path.join(RESULTS_DIR, 'losses',
                             'EURUSD_h05__qlike__seedmean.csv'),
                index_col=0, parse_dates=True)
print(L.shape, '(forecasts x models)')
display(L.head())

d = L['HAR-RV'] - L.drop(columns='HAR-RV')      # loss differentials vs the baseline
display(d.mean().sort_values(ascending=False).to_frame('mean loss saved vs HAR-RV'))

## Notes

* **Resuming** — a cell whose `.npz` is on Drive is skipped, so re-running step
  7 continues where it stopped. A cell that failed is recorded in
  `failures.csv` and skipped too; `--retry_failed` re-runs those.
* **Hyper-parameters** are the Optuna winners for **EUR/USD at h = 1**, applied
  to every dataset and horizon. That is a transfer, not a per-cell search — it
  is the fair-comparison choice, and it belongs in the caption of any table
  built from these results. Per-dataset anchors are a drop-in: put
  `tuning/ProjectC_tuning/<dataset>/<Model>_best.json` in place.
* **What is scored** — the h-day forward mean of RV, which is HAR-RV's target
  `Y^(h)`. The scale comes from the anchors: they carry `--log` today, so the
  sweep is `ln_RV`; a study tuned without it makes the whole pipeline raw. Every model in a block is scored against the same
  actuals, rebuilt from the CSV in float64, and each cell's deviation from them
  is reported as `target_dev` in `metrics.csv`.
* **Cost** — the cheap models (FITS, DLinear, TSLANet) are seconds per cell;
  TimesNet and MSGNet dominate. Expect an overnight run on a T4 for the full
  grid.
* Details, file formats and the two `exp/` changes this work made:
  `orchestrate/README.md`.